In [39]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException



# Uncomment the line below to run Chrome in headless mode
# # chrome_options.add_argument("--headless")
driver = webdriver.Chrome()


driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")

wait = WebDriverWait(driver, 20)

advance_btn = wait.until(
EC.element_to_be_clickable(
    (By.XPATH, "//button[contains(., 'Show Advance Search')]")
    )
    )

advance_btn.click()

In [40]:
# -----------------------------
# Select Major Clearance Type
# -----------------------------
major_clearance = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='majorClearanceType']")
    )
)

Select(major_clearance).select_by_value("1")


In [41]:
dropdown_element = driver.find_element(By.CSS_SELECTOR, "select[formcontrolname='issueAuthority']")
# 2. Wrap it in Selenium's Select class
select = Select(dropdown_element)

# 3. Select "SEIAA" by its value attribute
select.select_by_value("SEIAA")

In [42]:
# -----------------------------
# Select State
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='state']")
    )
)
# Fixed: changed state_dropdown_element to state_dropdown to match your variable above
state_select = Select(state_dropdown)

options = [opt.get_attribute("value") for opt in state_select.options if opt.get_attribute("value") != ""]



In [43]:
# --- Initialize data accumulation storage BEFORE the loop ---
all_table_data = []
headers = []  # We will capture headers on the first successful pass

# -----------------------------
# 1. Get the total number of options first safely
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
)
total_options = len(Select(state_dropdown).options)

# -----------------------------
# 2. Loop by index
# -----------------------------
for index in range(1, total_options):  # Start at 1 to skip "Select State" placeholder
    
    # FIX A: Wait for the dropdown to be fully present and visible
    state_dropdown = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
    )
    state_select = Select(state_dropdown)
    
    # Safely pull the value for debugging text
    state_value = state_select.options[index].get_attribute("value")
    
    # Select by index
    state_select.select_by_index(index)
    print(f"\nProcessing state index {index} (Value: {state_value})...")

    # -----------------------------
    # Click Search Button Safely
    # -----------------------------
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
    )
    
    # Track old table layout
    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    # FIX: Use JavaScript execution to bypass UI interception blocks
    driver.execute_script("arguments[0].click();", search_button)
    print(f"Search successfully forced click for: {state_value}")
    
    # -----------------------------
    # Wait for table to completely update
    # -----------------------------
    if old_table:
        try:
            # Wait for the old table to break/disappear as Angular updates the DOM
            wait.until(EC.staleness_of(old_table))
        except Exception:
            # Fallback if it transitions too instantly to catch staleness
            time.sleep(1) 

# -----------------------------
    # FIX: Handle Missing Table if No Results Exist
    # -----------------------------
    try:
        # Give it a shorter explicit wait (e.g., 4 seconds) to appear
        wait.until(
            EC.visibility_of_element_located((By.ID, "excel-table"))
        )
        print(f"Table successfully loaded for state: {state_value}")
    except TimeoutException:
        # This will now work perfectly because TimeoutException is imported!
        print(f"⚠️ No results found (Timeout) for state: {state_value}. Skipping...")
        time.sleep(1)
        continue  # Breaks out of current loop iteration, advances to next state item
    except Exception as e:
        # Catch-all backup just in case something else weird happens
        print(f" An unexpected error occurred on state {state_value}: {e}. Skipping...")
        time.sleep(1)
        continue

    # -----------------------------
    # SCRAPING LOGIC 
    # -----------------------------
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find("table", {"id": "excel-table"})
    
    if table:
        tbody = table.find("tbody")
        if tbody:
            rows = tbody.find_all("tr")
            
            # Secondary Check: Look for text indicators like "No Records Found" inside tbody
            tbody_text = tbody.get_text(strip=True).lower()
            if "no record" in tbody_text or "no data" in tbody_text or not rows:
                print(f"ℹ️ Table contains explicit empty notice for state: {state_value}. Skipping...")
                time.sleep(1)
                continue

            # Extract headers (Only run this once if headers are empty)
            if not headers:
                thead = table.find("thead")
                if thead:
                    for th in thead.find_all("th"):
                        headers.append(th.get_text(strip=True))

            # Extract rows for this specific state
            for row in rows:
                cols = row.find_all("td")
                row_data = [col.get_text(" ", strip=True) for col in cols]
                
                # Append the state name/value to each row so you know where it came from
                row_data.append(state_value) 
                all_table_data.append(row_data)

    # FIX C: Give Angular a quick breather to finish processing before jumping back 
    time.sleep(1)

# -----------------------------
# 3. Post-Loop Data Compilation
# -----------------------------
if all_table_data:
    # Append a 'State_Value' header since we added it to our rows above
    if headers and len(headers) < len(all_table_data[0]):
        headers.append("State_Value")
        
    # Convert all gathered rows across ALL states into one massive DataFrame
    df = pd.DataFrame(all_table_data, columns=headers if headers else None)
    print("\n--- Final Extracted Dataset ---")
    print(df)

    # Save everything into a single CSV file without overwriting
    df.to_csv("parivesh_data.csv", index=False)
    print("\nAll available states scraped and saved to parivesh_data.csv successfully!")
else:
    print("\n❌ Automation complete. Zero data entries found across all processed states.")


Processing state index 1 (Value: 35)...
Search successfully forced click for: 35
Table successfully loaded for state: 35

Processing state index 2 (Value: 28)...
Search successfully forced click for: 28
Table successfully loaded for state: 28

Processing state index 3 (Value: 12)...
Search successfully forced click for: 12
Table successfully loaded for state: 12

Processing state index 4 (Value: 18)...
Search successfully forced click for: 18
Table successfully loaded for state: 18

Processing state index 5 (Value: 10)...
Search successfully forced click for: 10
Table successfully loaded for state: 10

Processing state index 6 (Value: 4)...
Search successfully forced click for: 4
Table successfully loaded for state: 4

Processing state index 7 (Value: 22)...
Search successfully forced click for: 22
Table successfully loaded for state: 22

Processing state index 8 (Value: 7)...
Search successfully forced click for: 7
Table successfully loaded for state: 7

Processing state index 9 (Val

In [44]:
# Quit the WebDriver
driver.quit()

In [ ]:
# imports

import requests
from IPython.display import Markdown, display
import ollama

In [ ]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)
    
pr = Website("https://parivesh.nic.in/")

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

# And now: call the Ollama function instead of OpenAI

# Constants

MODEL = "llama3.2"

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']


# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

summarize("https://parivesh.nic.in/")
display_summary("https://anthropic.com")

In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

display(response.choices[0].message.content)
def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']
chrome_options = webdriver.ChromeOptions()